Prepare

In [ ]:
!apt-get install -y aria2
%cd /content
![ -d Fooocus ] || git clone https://github.com/parsamrrelax/Fooocus.git
%cd /content/Fooocus

Hugging Face token (optional)

In [ ]:
#@markdown Paste a [Hugging Face access token](https://huggingface.co/settings/tokens) to speed up model downloads and avoid Hugging Face's anonymous rate limit. Leave blank to download the same way as before.
hf_token = "" #@param {type:"string"}

import os
if hf_token.strip():
    os.environ["HF_TOKEN"] = hf_token.strip()
    print("HF_TOKEN set, downloads will use it to speed up and avoid rate limits.")
else:
    os.environ.pop("HF_TOKEN", None)
    print("No HF_TOKEN provided, downloads will use the default (unauthenticated) method.")

Fix Gradio WebSocket (error 1006)

In [ ]:
# Patch Gradio WebSocket limits (error 1006 on large images). Edits the installed
# file on disk before launch starts a fresh Python process — no Colab runtime restart.
!pip install -q "gradio==3.41.2"

from pathlib import Path
import gradio
import re

path = Path(gradio.__file__).resolve().parent / "networking.py"
text = path.read_text()
marker = "ws_max_size=1 * 1024 * 1024 * 1024"
if marker in text:
    print(f"Already patched: {path}")
else:
    replacement = """config = uvicorn.Config(
        app=app,
        port=port,
        host=host,
        log_level="warning",
        ssl_keyfile=ssl_keyfile,
        ssl_certfile=ssl_certfile,
        ssl_keyfile_password=ssl_keyfile_password,
        ws_max_size=1 * 1024 * 1024 * 1024,  # Setting max websocket size to be 1 GB
        ws_max_queue=64,
        ws_ping_interval=60.0,
        ws_ping_timeout=10.0,
        ws_per_message_deflate=False,
        reload=True,
        timeout_notify=120
    )"""
    pattern = re.compile(
        r"config\s*=\s*uvicorn\.Config\(\s*"
        r"app=app,\s*"
        r"port=port,\s*"
        r"host=host,\s*"
        r'log_level="warning",\s*'
        r"ssl_keyfile=ssl_keyfile,\s*"
        r"ssl_certfile=ssl_certfile,\s*"
        r"ssl_keyfile_password=ssl_keyfile_password,\s*"
        r".*?"
        r"\)",
        re.DOTALL,
    )
    new_text, n = pattern.subn(replacement, text, count=1)
    if n != 1:
        raise RuntimeError(f"Could not find uvicorn.Config(...) to patch in {path}")
    path.write_text(new_text)
    print(f"Patched Gradio WebSocket limits in {path}")


Run

In [ ]:
!python entry_with_update.py --share --always-high-vram --disable-offload-from-vram